# Lab 05.3 — Invoke Runtime with SSE Streaming

## Overview

Login da Ana → invoca o SmartAgent → vê o stream SSE → ela is roteada
automaticamente para o specialist apropriado.

## Prerequisites

- ✅ Lab 05.2 (todos 6 runtimes deployados)

## Setup

In [ ]:
import sys
import importlib.util
sys.path.insert(0, "..")
from shared.utils.config import load_config, get_region
from utils import invoke_runtime

cfg = load_config()
region = get_region()

# Carrega get_bearer_token do Lab 01
spec = importlib.util.spec_from_file_location("identity_utils", "../01-Identity-Foundation/utils.py")
identity_utils = importlib.util.module_from_spec(spec)
spec.loader.exec_module(identity_utils)

## Step 1: Login da Ana

In [ ]:
ana_tokens = identity_utils.get_bearer_token(
    pool_id=cfg["COGNITO_USER_POOL_ID"],
    client_id=cfg["COGNITO_CLIENT_ID"],
    username="ana.operadora@workshop.local",
    password="Workshop@2025!",
    region=region,
)
ana_token = ana_tokens["access_token"]

## Step 2: Pergunta sobre rede — SmartAgent deve rotear para GridMonitorAgent

In [ ]:
prompt = "Como is o east sector agora?"
print(f"Pergunta: {prompt}\n")

response = invoke_runtime(
    runtime_arn=cfg["RUNTIME_SMART_AGENT_ARN"],
    prompt=prompt,
    bearer_token=ana_token,
    region=region,
)
print(response)

## Step 3: Pergunta sobre fatura — SmartAgent deve rotear para BillingAgent

Mas Ana is operadora — Cedar P4 vai negar. Veremos o specialist devolver
a mensagem de erro.

In [ ]:
prompt = "Quero ver a fatura INV-2024-03-0091"
print(f"Pergunta: {prompt}\n")

response = invoke_runtime(
    runtime_arn=cfg["RUNTIME_SMART_AGENT_ARN"],
    prompt=prompt,
    bearer_token=ana_token,
    region=region,
)
print(response)

## Step 4: Login do Carlos (gestor) — pode aprovar

In [ ]:
carlos_tokens = identity_utils.get_bearer_token(
    pool_id=cfg["COGNITO_USER_POOL_ID"],
    client_id=cfg["COGNITO_CLIENT_ID"],
    username="carlos.gestor@workshop.local",
    password="Workshop@2025!",
    region=region,
)

prompt = "Aprove a work order WO-2024-0041"
print(f"Carlos pergunta: {prompt}\n")
response = invoke_runtime(
    runtime_arn=cfg["RUNTIME_SMART_AGENT_ARN"],
    prompt=prompt,
    bearer_token=carlos_tokens["access_token"],
    region=region,
)
print(response)

## 🎓 What you learned

- SmartAgent rota dinâmica via @tool delegations (Strands)
- JWT propaga em hops HTTPS (User → SmartAgent → Specialist → Gateway)
- Cedar enforce no Gateway is transparente para o agente — Specialist
  recebe AccessDeniedException e devolve mensagem amigável

## Cleanup

```python
from utils import cleanup_runtime
cleanup_runtime(cfg["RUNTIME_SMART_AGENT_ARN"], region=region)
# ... + os 5 specialists
```

## Next

➡️ [Lab 06 — Bedrock Guardrails](../06-Bedrock-Guardrails/)